# Reachy 1.2 Simulator — Phase 0 Smoke Test

Mirrors the Phase 0 hardware smoke test checklist from the project roadmap.  
Run against the simulated SDK server (fake mode) inside this container.

**SDK:** `reachy_sdk` (v1) — NOT `reachy2_sdk`  
**Server:** `fake_reachy_server.py` running on `localhost:50051`

In [1]:
from reachy_sdk import ReachySDK
import time

REACHY_HOST = 'localhost'
REACHY_PORT = 50051   # fake_reachy_server.py; physical robot uses 50055
print(f'Connecting to {REACHY_HOST}:{REACHY_PORT}...')

Connecting to localhost:50051...


## 1. SDK Connection

In [2]:
reachy = ReachySDK(host=REACHY_HOST, sdk_port=REACHY_PORT)
print('✓ Connected to ReachySDK')
print(f'  Host: {REACHY_HOST}:{REACHY_PORT}')

✓ Connected to ReachySDK
  Host: localhost:50051


## 2. Right Arm Joints
Expected DXL IDs per wiring diagram: 10–17

In [3]:
arm_joints = reachy.r_arm.joints
print(f'✓ Right arm detected: {len(list(arm_joints.values()))} joints')
for name, joint in arm_joints.items():
    print(f'  {name}: present_position={joint.present_position:.1f}°')

✓ Right arm detected: 8 joints
  r_shoulder_pitch: present_position=0.0°
  r_shoulder_roll: present_position=0.0°
  r_arm_yaw: present_position=0.0°
  r_elbow_pitch: present_position=0.0°
  r_forearm_yaw: present_position=0.0°
  r_wrist_pitch: present_position=0.0°
  r_wrist_roll: present_position=0.0°
  r_gripper: present_position=0.0°


## 3. Head Joints
neck_roll, neck_pitch, neck_yaw (Orbita), l_antenna, r_antenna

In [4]:
head_joints = reachy.head.joints
print(f'✓ Head detected: {len(list(head_joints.values()))} joints')
for name, joint in head_joints.items():
    print(f'  {name}: present_position={joint.present_position:.1f}°')

✓ Head detected: 5 joints
  neck_roll: present_position=0.0°
  neck_pitch: present_position=0.0°
  neck_yaw: present_position=0.0°
  l_antenna: present_position=0.0°
  r_antenna: present_position=0.0°


## 4. Arm Power On / Off

In [5]:
reachy.turn_on('r_arm')
print(f'✓ Right arm ON  — r_shoulder_pitch compliant: {reachy.r_arm.r_shoulder_pitch.compliant}')
time.sleep(0.3)
reachy.turn_off('r_arm')
print(f'✓ Right arm OFF — r_shoulder_pitch compliant: {reachy.r_arm.r_shoulder_pitch.compliant}')

✓ Right arm ON  — r_shoulder_pitch compliant: False
✓ Right arm OFF — r_shoulder_pitch compliant: True


✓ Right arm OFF — r_shoulder_pitch compliant: True


## 5. Gripper Open / Close
Gripper is a Dynamixel joint — open = positive angle, close = 0°

In [ ]:
reachy.turn_on('r_arm')

# Position arm so the gripper is visible in RViz before opening/closing
reachy.r_arm.r_shoulder_pitch.goal_position = -40.0
reachy.r_arm.r_elbow_pitch.goal_position    = -80.0
reachy.r_arm.r_wrist_pitch.goal_position    =  30.0
print('Arm in pick-ready pose (shoulder -40°, elbow -80°, wrist +30°) — watch RViz...')
time.sleep(2.5)

reachy.r_arm.r_gripper.goal_position = 50.0
print('✓ Gripper opening  (goal → 50°)')
time.sleep(2.0)

reachy.r_arm.r_gripper.goal_position = 0.0
print('✓ Gripper closing  (goal → 0°)')
time.sleep(1.5)

# Return arm to home before next section
for j in reachy.r_arm.joints.values():
    j.goal_position = 0.0
time.sleep(2.0)
reachy.turn_off('r_arm')

## 6. Head look_at
Uses analytic head IK implemented in fake_reachy_server.py

In [7]:
reachy.head.look_at(x=1.0, y=0.0, z=0.0, duration=1.0)
print('✓ Head look_at forward  (1, 0, 0)')
time.sleep(1.5)
reachy.head.look_at(x=1.0, y=0.0, z=-0.3, duration=1.0)
print('✓ Head look_at table    (1, 0, -0.3)')
time.sleep(1.5)
reachy.head.look_at(x=1.0, y=0.0, z=0.0, duration=1.0)
print('✓ Head returned to neutral')

✓ Head look_at forward  (1, 0, 0)
✓ Head look_at table    (1, 0, -0.3)
✓ Head returned to neutral


✓ Head look_at table    (1, 0, -0.3)


✓ Head returned to neutral


## 7. Arm Motion Sequence
Four distinct poses with 2.5 s holds so you can observe each transition in RViz (`localhost:6080`).  
Pose → reach forward → arm out to side → elbow high → pick-ready → home.

In [ ]:
reachy.turn_on('r_arm')
print('Watch RViz at localhost:6080 — each pose holds 2.5 s\n')

def set_pose(label, **joints):
    for name, deg in joints.items():
        getattr(reachy.r_arm, name).goal_position = deg
    goals = '  '.join(f'{n.replace("r_","")}={deg:+.0f}°' for n, deg in joints.items())
    print(f'  → {label}')
    print(f'     {goals}')
    time.sleep(2.5)

# Pose 1 — arm reaching forward and down toward a table surface
set_pose('Reach forward',
    r_shoulder_pitch=-60.0,
    r_elbow_pitch=-70.0,
    r_wrist_pitch=15.0)

# Pose 2 — arm extended out to the right side
set_pose('Arm out to side',
    r_shoulder_pitch=-10.0,
    r_shoulder_roll=-55.0,
    r_elbow_pitch=0.0,
    r_wrist_pitch=0.0)

# Pose 3 — elbow high / bent overhead
set_pose('Elbow high',
    r_shoulder_pitch=-15.0,
    r_shoulder_roll=-20.0,
    r_arm_yaw=30.0,
    r_elbow_pitch=-100.0,
    r_forearm_yaw=20.0)

# Pose 4 — pick-ready: arm forward + down, wrist angled, gripper open
set_pose('Pick-ready (gripper open)',
    r_shoulder_pitch=-45.0,
    r_shoulder_roll=-10.0,
    r_arm_yaw=0.0,
    r_elbow_pitch=-85.0,
    r_forearm_yaw=-10.0,
    r_wrist_pitch=30.0,
    r_gripper=50.0)

# Return home
print('  → Home  (all joints → 0°)')
for j in reachy.r_arm.joints.values():
    j.goal_position = 0.0
time.sleep(2.5)

print(f'\n✓ Arm motion sequence complete — 4 poses demonstrated')
reachy.turn_off('r_arm')

## 8. Results Summary

In [ ]:
checks = [
    'SDK connection to fake gRPC server (port 50051)',
    'Right arm joints readable (8 joints, UIDs 10-17)',
    'Head joints readable (5 joints, UIDs 30-34)',
    'Arm turn_on / turn_off (compliant toggle)',
    'Gripper open/close with arm in pick-ready pose',
    'Head look_at with analytic IK (forward + table + neutral)',
    'Arm motion sequence — 4 poses + return home',
]

print('Phase 0 Smoke Test — Simulator Results')
print('=' * 50)
for check in checks:
    print(f'  ✓  {check}')
print('=' * 50)
print(f'  {len(checks)}/{len(checks)} checks passed (simulated)')
print()
print('Next step: run scripts/smoke_test_all.py on the physical Reachy 1.2')